In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array, array_to_vector

# Models:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.classification import MultilayerPerceptronClassifier

In [0]:
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="recallByLabel"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel"
)

In [0]:
train=spark.read.table("teams.sensorx.train_data")
test=spark.read.table("teams.sensorx.test_data")

n_lags = 20
H_days = 15

## Gradient Boosted Trees Model Training and Evaluation

In [0]:
# Class weighting (give minority class higher weight)
label_counts = train.groupBy("label").count().collect()
total = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total / (2.0 * row["count"]) for row in label_counts}

for lbl in sorted(weight_map):
    cnt = [r["count"] for r in label_counts if r["label"] == lbl][0]
    pct = cnt / total * 100
    print(f"  Label {lbl}: {cnt:,} samples ({pct:.1f}%) -> weight = {weight_map[lbl]:.4f}")
print(f"  Total: {total:,} samples")

# Add weight column to train and test
weight_expr = F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])
train_w = train.withColumn("weight", weight_expr)
test_w = test.withColumn("weight", weight_expr)

In [0]:
# Strip corrupted ML metadata from features vector (fixes issues with Spark ML models after saving/loading)
train_gbt = train_w.withColumn("features", array_to_vector(vector_to_array("features")))
test_gbt = test_w.withColumn("features", array_to_vector(vector_to_array("features")))

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=30,
    maxBins=64,
    seed=42
)

GBTmodel = gbt.fit(train_gbt)
gbtpredictions = GBTmodel.transform(test_gbt)

# --- Training accuracy ---
gbt_train_preds = GBTmodel.transform(train_gbt)
GBT_train_auc = auc_eval.evaluate(gbt_train_preds)

# --- Test evaluation ---
GBT_auc = auc_eval.evaluate(gbtpredictions)

# Recall (per class)
GBT_recall_0 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 0.0})
GBT_recall_1 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
GBT_f1_0 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 0.0})
GBT_f1_1 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nGBT with {n_lags} lags + deviceId (class-weighted)")
print(f"Training AUC: {GBT_train_auc:.4f}")
print(f"Test AUC:     {GBT_auc:.4f}")
print(f"Recall (label=0): {GBT_recall_0:.4f}")
print(f"Recall (label=1): {GBT_recall_1:.4f}")
print(f"F1     (label=0): {GBT_f1_0:.4f}")
print(f"F1     (label=1): {GBT_f1_1:.4f}")

# Confusion matrix
gbtpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

## Random Forest Model Training and Evaluation


In [0]:
# --- RandomForest with class weighting ---
# Strip corrupted ML metadata from features vector (same fix as GBT)
from pyspark.ml.functions import vector_to_array, array_to_vector

n_lags = 20
train_rf = train_w.withColumn("features", array_to_vector(vector_to_array("features")))
test_rf = test_w.withColumn("features", array_to_vector(vector_to_array("features")))

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    maxBins=64,
    seed=42
)
RFmodel = rf.fit(train_rf)
RFpredictions = RFmodel.transform(test_rf)

RF_train_auc = auc_eval.evaluate(RFmodel.transform(train_rf))
RF_auc = auc_eval.evaluate(RFpredictions)

# Recall (per class)
RF_recall_0 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 0.0})
RF_recall_1 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
RF_f1_0 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 0.0})
RF_f1_1 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nRandom Forest with {n_lags} lags + deviceId (class-weighted)")
print(f"Training AUC: {RF_train_auc:.4f}")
print(f"Test AUC:     {RF_auc:.4f}")
print(f"Recall (label=0): {RF_recall_0:.4f}")
print(f"Recall (label=1): {RF_recall_1:.4f}")
print(f"F1     (label=0): {RF_f1_0:.4f}")
print(f"F1     (label=1): {RF_f1_1:.4f}")

# Confusion matrix
RFpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

## Multilayer Perceptron Classifier Model Training and Evaluation

In [0]:
# --- Normalize features for MLP ---
# Fit scaler on FULL train (preserves accurate mean/std statistics)
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(train)

# Transform full train and full test
train_norm = scaler_model.transform(train).drop("features").withColumnRenamed("features_scaled", "features")
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

print(f"\nTraining set size: {train_norm.count():,}")
print(f"Test set size: {test_norm.count():,}")
print(f"Normalized feature vector size: {train_norm.select('features').first()[0].size}")

In [0]:
# Input layer = 112 features (5 sensor lagged + 100 lags + 6 deviation + 1 deviceId, NO onTime)
layers = [112, 16, 16, 2]
H_days = 15
n_lags = 20

trainer = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    layers=layers,
    blockSize=128,
    seed=42,
    stepSize=0.001
)

MLPmodel = trainer.fit(train_norm)
MLPpredictions = MLPmodel.transform(test_norm)

# --- Evaluate ---
MLP_training_auc = auc_eval.evaluate(MLPmodel.transform(train_norm))
MLP_auc = auc_eval.evaluate(MLPpredictions)

MLP_recall_0 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 0.0})
MLP_recall_1 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 1.0})
MLP_f1_0 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 0.0})
MLP_f1_1 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nMLP Neural Network (15-day horizon, 112 features, onTime REMOVED)")
print(f"Layers: {layers}")
print(f"Training AUC: {MLP_training_auc:.4f}")
print(f"Test AUC:     {MLP_auc:.4f}")
print(f"  Class 0 (safe):  Recall={MLP_recall_0:.4f}  F1={MLP_f1_0:.4f}")
print(f"  Class 1 (fault): Recall={MLP_recall_1:.4f}  F1={MLP_f1_1:.4f}")

print("\nConfusion matrix:")
MLPpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
import os
import mlflow
from mlflow.models import infer_signature
from pyspark.ml.functions import vector_to_array

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/sensorx_mlp_fault_prediction")

CATALOG = "teams"
SCHEMA = "sensorx"
MLP_MODEL_NAME = f"{CATALOG}.{SCHEMA}.mlp_fault_prediction"

with mlflow.start_run(run_name="MLP_fault_prediction_v7_no_onTime") as run:
    mlflow.log_param("model_type", "MultilayerPerceptronClassifier")
    mlflow.log_param("layers", str(layers))
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("stepSize", 0.001)
    mlflow.log_param("blockSize", 128)
    mlflow.log_param("n_lags", n_lags)
    mlflow.log_param("H_days", H_days)
    mlflow.log_param("normalized", True)
    mlflow.log_param("deviation_features", True)
    mlflow.log_param("onTime_included", False)
    mlflow.log_param("delta_seconds_lagged", True)
    mlflow.log_param("num_features", 112)
    mlflow.log_param("description", "5 sensors lagged (20), onTime REMOVED, 6 deviation, 1 deviceId")

    mlflow.log_metric("AUC", MLP_auc)
    mlflow.log_metric("Training_AUC", MLP_training_auc)
    mlflow.log_metric("Recall_label_0", MLP_recall_0)
    mlflow.log_metric("Recall_label_1", MLP_recall_1)
    mlflow.log_metric("F1_label_0", MLP_f1_0)
    mlflow.log_metric("F1_label_1", MLP_f1_1)

    sample_input = test_norm.select("features").limit(5)
    sample_output = MLPmodel.transform(sample_input).select("prediction")
    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    input_example = (
        sample_input.limit(1)
        .select(vector_to_array("features").alias("features"))
        .toPandas()
    )

    model_info = mlflow.spark.log_model(
        MLPmodel,
        artifact_path="mlp_model",
        signature=signature,
        input_example=input_example,
        registered_model_name=MLP_MODEL_NAME,
    )

    print(f"Model registered to: {MLP_MODEL_NAME}")
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"Model version: {model_info.registered_model_version}")
    print(f"Test AUC: {MLP_auc:.4f}")
    print(f"Recall (label=1): {MLP_recall_1:.4f}")
    print(f"F1 (label=1): {MLP_f1_1:.4f}")